In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import Literal

MODEL_NAME = "stabilityai/stable-code-3b"

def get_best_device() -> Literal["mps", "cpu", "cuda"]:
    """
    Returns the best available torch device:
    Priority: CUDA > MPS > CPU
    """
    
    if torch.cuda.is_available():
        return torch.device("cuda")
    
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps") # mac only
    
    return torch.device("cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # necessary for training, since this model does not have a dedicated pad_token

model = AutoModelForCausalLM.from_pretrained(
  MODEL_NAME,
  torch_dtype="auto",
)
model.to(get_best_device())

# Tokenizer & Dataset

In [ ]:
from datasets import load_dataset

data_files = {
    "train": "./data/train.parquet",
    "test": "./data/test.parquet",
}

# Load your local parquet file
dataset = load_dataset("parquet", data_files=data_files)

# Inspect the structure
print(dataset["train"][0])

In [ ]:

def tokenize_function(examples):
    # We only want the 'content' column
    return tokenizer(
        examples["content"], 
        truncation=True, 
        max_length=1024
    )

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True
)

In [ ]:
# Test if tokenization worked

sample = tokenized_datasets["train"][0]
decoded_text = tokenizer.decode(sample["input_ids"])

print("---- DECODED TEXT ----")
print(decoded_text[:500])
print("\n---- RAW INPUT IDS ----")
print(sample["input_ids"][:20])

# Training

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./code-completion-model",
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=1, 
    gradient_accumulation_steps=16, 
    eval_steps=200,               
    save_steps=400,
    logging_steps=50,
    learning_rate=5e-5,           
    weight_decay=0.01,
    fp16=False,                   
    push_to_hub=False,            
    report_to="none"              # Can change to "wandb" or "tensorboard"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
)

trainer.train()